In [1]:
import modified_didppy as m_dp
import vrplib
import numpy as np
import math
import tempfile
import os
from scipy.spatial.distance import cdist
import pulp
import re

# **Data**

## **Dataset A**

### Load the data

In [2]:
# 1. Define your directory path
# TIP: Use r"..." (raw string) so Python treats backslashes as text, not escape characters
base_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_CVRP_dual_bounds_and_models\Datasets\A"

# 2. Construct the full file paths
# We assume the solution file has the standard .sol extension
instance_path = os.path.join(base_path, "A-n32-k5.vrp")
solution_path = os.path.join(base_path, "A-n32-k5.sol")

# 3. Load the data
try:
    # Read the instance data
    instance = vrplib.read_instance(instance_path)
    
    # Read the solution data
    solution = vrplib.read_solution(solution_path)

    # 4. Print results to verify
    print(f"Successfully loaded: {instance['name']}")
    print(f"Dimension: {instance['dimension']}")
    print(f"Vehicle Capacity: {instance['capacity']}")
    print("-" * 20)
    print(f"Optimal Cost (from solution): {solution['cost']}")
    print(f"Routes: {solution['routes']}")

except FileNotFoundError:
    print("Error: Could not find the file. Please check if 'A-n53-k7.sol' exists in that folder.")

Successfully loaded: A-n32-k5
Dimension: 32
Vehicle Capacity: 100
--------------------
Optimal Cost (from solution): 784
Routes: [[21, 31, 19, 17, 13, 7, 26], [12, 1, 16, 30], [27, 24], [29, 18, 8, 9, 22, 15, 10, 25, 5, 20], [14, 28, 11, 4, 23, 3, 2, 6]]


### Extract data

In [3]:
# 'instance' is the dictionary you provided in the prompt
# 1. Extract Constraints
capacity = instance['capacity']
num_locations = instance['dimension']
match = re.search(r"No of trucks:\s*(\d+)", instance['comment'])
if match:
    num_vehicles = int(match.group(1))
else:
    print("Number of trucks not found.")
    
# 2. Extract Demands
# Note: instance['demand'] includes the Depot at index 0 (value 0)
# This is usually what you want for 0-based indexing in state representation
cust_demands = instance['demand'] 

# 3. Extract and FIX the Distance Matrix
# The library calculated exact Euclidean distances (floats). 
# For the 'A' (Augerat) series, we typically round to the nearest integer.
travel_cost = instance['edge_weight']


# --- VERIFICATION ---
print(f"Capacity: {capacity}")
print(f"Number of Nodes: {num_locations}")
print(f"Number of Vehicles: {num_vehicles}")
print(f"Depot Demand: {cust_demands[0]}")
print(f"Customer 1 Demand: {cust_demands[1]}")

print("\nComparison of Distance (Depot -> Node 1):")
print(f"Distance matrix: {travel_cost[0][1]}") 

Capacity: 100
Number of Nodes: 32
Number of Vehicles: 5
Depot Demand: 0
Customer 1 Demand: 19

Comparison of Distance (Depot -> Node 1):
Distance matrix: 34.92849839314596


# **CVRP relax model**

In [4]:
def create_vrp4_relaxed_model(num_locations, num_vehicles, capacity, cust_demands, travel_cost):
    """
    Creates a 3-Index Vehicle Flow (VRP4) Linear Programming Relaxation model.
    Based on Toth & Vigo (2002) formulation with MTZ subtour elimination constraints.
    
    The binary variables x_ijk and y_ik are relaxed to continuous [0, 1].
    """
    # Map input argument name to the internal name used in your logic
    demands = cust_demands 

    # --- 1. Initialize Model ---
    # In Pulp, we define the problem and the sense (Minimize)
    mdl = pulp.LpProblem("VRP4_Relaxed_3Index", pulp.LpMinimize)
    
    # Optimization Parameters (Note: Pulp handles solver parameters during the .solve() call, 
    # but we initialize the structure here just like the Docplex version)

    # --- 2. Create Variables (Relaxed to Continuous) ---
    
    # x[i, j, k]: Fraction of traversal from node i to node j by vehicle k
    # Relaxed from Binary {0,1} to Continuous [0,1]
    # Corresponds to (1.35) in Toth & Vigo
    x = {}
    for k in range(num_vehicles):
        for i in range(num_locations):
            for j in range(num_locations):
                if i != j:
                    x[(i, j, k)] = pulp.LpVariable(f"x_{i}_{j}_{k}", lowBound=0, upBound=1, cat=pulp.LpContinuous)

    # y[i, k]: Fraction of service for customer i by vehicle k
    # Relaxed from Binary {0,1} to Continuous [0,1]
    # Corresponds to (1.34) in Toth & Vigo
    y = {}
    for k in range(num_vehicles):
        for i in range(num_locations):
            y[(i, k)] = pulp.LpVariable(f"y_{i}_{k}", lowBound=0, upBound=1, cat=pulp.LpContinuous)

    # u[i, k]: MTZ auxiliary variable for vehicle k at node i
    # Represents accumulated load/time. Used to eliminate subtours.
    # Corresponds to (1.38) bounds: d_i <= u_ik <= C
    # Note: Only needed for customers (V \ {0})
    u = {}
    for k in range(num_vehicles):
        for i in range(1, num_locations): 
            u[(i, k)] = pulp.LpVariable(f"u_{i}_{k}", lowBound=demands[i], upBound=capacity, cat=pulp.LpContinuous)


    # --- 3. Objective Function (1.28) ---
    # Minimize Sum of Costs: sum(c_ij * x_ijk)
    obj_expr = pulp.lpSum(travel_cost[i][j] * x[(i, j, k)] 
                          for k in range(num_vehicles)
                          for i in range(num_locations)
                          for j in range(num_locations) if i != j)
    mdl += obj_expr, "Minimize_Total_Cost"

    # --- 4. Constraints ---

    # (1.29) Assignment Constraint: Each customer served exactly once
    # sum(y_ik over k) = 1 for all i in V\{0}
    for i in range(1, num_locations):
        mdl += (
            pulp.lpSum(y[(i, k)] for k in range(num_vehicles)) == 1,
            f"assign_cust_{i}"
        )

    # (1.31) Flow Conservation Constraint
    # sum(x_ijk over j) = y_ik  AND  sum(x_jik over j) = y_ik
    # If vehicle k serves i (y_ik > 0), it must enter i and leave i.
    for k in range(num_vehicles):
        for i in range(num_locations):
            # Inflow to i
            inflow = pulp.lpSum(x[(j, i, k)] for j in range(num_locations) if i != j)
            # Outflow from i
            outflow = pulp.lpSum(x[(i, j, k)] for j in range(num_locations) if i != j)
            
            mdl += (inflow == y[(i, k)], f"flow_in_{i}_{k}")
            mdl += (outflow == y[(i, k)], f"flow_out_{i}_{k}")

    # (1.32) Capacity Constraint
    # sum(d_i * y_ik) <= C for each vehicle k
    for k in range(num_vehicles):
        load_sum = pulp.lpSum(demands[i] * y[(i, k)] for i in range(1, num_locations)) 
        mdl += (load_sum <= capacity, f"capacity_veh_{k}")

    # (1.30) Fleet Constraint (Optional but standard)
    # Ensure exactly (or at most) K vehicles leave the depot.
    # sum(y_0k) <= K  (Using y[(0,k)] which tracks depot usage)
    mdl += (
        pulp.lpSum(y[(0, k)] for k in range(num_vehicles)) <= num_vehicles,
        "fleet_size"
    )

    # (1.37) Generalized MTZ Subtour Elimination Constraints
    # u_ik - u_jk + C * x_ijk <= C - d_j
    # This prevents cycles that do not include the depot.
    for k in range(num_vehicles):
        for i in range(1, num_locations):
            for j in range(1, num_locations):
                if i != j:
                    # Valid only if d_i + d_j <= C
                    if demands[i] + demands[j] <= capacity:
                        mdl += (
                            u[(i, k)] - u[(j, k)] + capacity * x[(i, j, k)] <= capacity - demands[j],
                            f"mtz_{i}_{j}_{k}"
                        )

    return mdl

# **Execution**

In [7]:
# --- Building VRP4 Relaxation Model ---
print("\n--- Building VRP4 Relaxation Model ---")
# Note: Ensure n_nodes, n_vehicles, etc. are defined before calling this
model = create_vrp4_relaxed_model(num_locations = num_locations, 
                                num_vehicles = num_vehicles, 
                                capacity = capacity, 
                                cust_demands = cust_demands, 
                                travel_cost = travel_cost)

# 4. Solve
print("--- Solving with CPLEX ---")

# In Pulp, we define the solver with parameters first
# msg=True enables the solver log output
solver = pulp.CPLEX_CMD(timeLimit=60, msg=True)

# We pass the solver to the solve() method
# If CPLEX is not installed, this line might throw an error. 
# You can try `mdl.solve()` to use the default solver (CBC) if CPLEX fails.
model.solve(solver)

# 5. Output Results
# Check if a valid solution was found (Optimal or Feasible)
if model.status == pulp.LpStatusOptimal or model.status == pulp.LpStatusNotSolved: 
    # Note: Sometimes 'NotSolved' in Pulp might still have intermediate values for relaxations,
    # but strictly speaking we look for Optimal/Feasible. 
    
    print("\n" + "="*40)
    print(f"LOWER BOUND (Objective): {pulp.value(model.objective):.4f}")
    print(f"Solve Status: {pulp.LpStatus[model.status]}")
    print("="*40)
    
    # Optional: Inspect flows to confirm fractional relaxation
    print("\nSignificant Fractional Flows (> 0.01):")
    
    # Pulp provides a dictionary of all variables {name: var_object}
    # This is much faster than iterating linearly to find a name
    var_dict = model.variablesDict()
    
    count = 0
    for k in range(num_vehicles):
        for i in range(num_locations):
            for j in range(num_locations):
                if i != j:
                    # Reconstruct the name string used in the definition
                    var_name = f'x_{i}_{j}_{k}'
                    
                    if var_name in var_dict:
                        var = var_dict[var_name]
                        val = var.varValue
                        
                        if val and val > 0.01:
                            print(f"Veh {k} | {i:2d} -> {j:2d} : {val:.2f}")
                            count += 1
                            if count > 5: break 
        if count > 5: break

else:
    print("\nNo solution found (Infeasible or Error)")
    print(f"Status: {pulp.LpStatus[model.status]}")


--- Building VRP4 Relaxation Model ---
--- Solving with CPLEX ---

LOWER BOUND (Objective): 351.9272
Solve Status: Optimal

Significant Fractional Flows (> 0.01):
Veh 0 |  1 -> 12 : 0.21
Veh 0 |  2 -> 23 : 0.21
Veh 0 |  3 ->  2 : 0.21
Veh 0 |  7 -> 16 : 0.18
Veh 0 | 12 ->  1 : 0.21
Veh 0 | 14 -> 24 : 0.24
Veh 0 | 16 ->  7 : 0.18
Veh 0 | 21 -> 31 : 0.12
Veh 0 | 23 ->  3 : 0.21
Veh 0 | 24 -> 27 : 0.24
Veh 0 | 27 -> 14 : 0.24
Veh 0 | 31 -> 21 : 0.12
